# Discount A/B test - SYNTHETIC experiment analysis
> **Everything outcome-related in this notebook is simulated.** Olist contains no randomized promotion. Historical data only calibrates the control arm (baseline conversion via a back-test, order values via resampling). The treatment effect is an assumption.

In [1]:
import sys, json; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.db import query
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.prop_cycle': plt.cycler(color=['#2a78d6', '#eb6834', '#1baf7a'])})
from dataclasses import replace
from src.experiment_simulator import ExperimentConfig, load_calibration
from src.experiment_analysis import analyze, monte_carlo, sample_size, break_even

## 1. Calibration (observed back-test at 2018-07-01)

In [2]:
cal = load_calibration(); {k: v for k, v in cal.items() if k != 'order_values'}

{'snapshot': '2018-07-01',
 'window_days': 60,
 'backtest_eligible': 61375,
 'backtest_converted': 192,
 'baseline_conversion': 0.0031283095723014256,
 'eligible_population': 65627,
 'extra_orders_rate': 0.050872093023255814,
 'source': 'Historical Olist back-test (observed). Used only to calibrate the control arm.'}

## 2. Pre-registered plan
* Primary metric: contribution profit per eligible customer (60 days)
* Secondary: conversion rate; guardrails: AOV, discount cost
* α = 0.05 two-sided, target power 80%

In [3]:
cfg = ExperimentConfig(); r = analyze(cfg, cal); r['config']

{'n_eligible': 65627,
 'treatment_share': 0.5,
 'discount_pct': 0.1,
 'baseline_conversion': 0.0031283095723014256,
 'price_elasticity': 1.5,
 'aov_uplift_treated': 0.0,
 'contribution_margin': 0.15,
 'variable_cost_per_order': 1.5,
 'window_days': 60,
 'seed': 42,
 'extra_orders_rate': 0.050872093023255814,
 'relative_lift_assumed': 0.15000000000000002}

In [4]:
r['groups']

group,control,treatment
eligible_customers,32813.000000,32814.000000
conversions,103.000000,114.000000
conversion_rate,0.003139,0.003474
orders,111.000000,118.000000
gmv,15235.120000,15156.140000
aov_gmv,137.253333,128.441864
gmv_per_eligible,0.464301,0.461880
net_revenue_per_eligible,0.464301,0.415692
discount_cost_per_eligible,0.000000,0.046188
contribution_per_eligible,0.064571,0.017700


In [5]:
r['conversion']

{'control_rate': 0.0031389997866699175,
 'treatment_rate': 0.003474126897056135,
 'control_conversions': 103,
 'treatment_conversions': 114,
 'n_control': 32813,
 'n_treatment': 32814,
 'abs_lift': 0.00033512711038621726,
 'abs_lift_ci': [np.float64(-0.0005432965321423695),
  np.float64(0.0012135507529148042)],
 'rel_lift': 0.1067623871175043,
 'rel_lift_ci': [-0.15173812332918557, 0.4440387045867358],
 'z': 0.7477414441893828,
 'p_value': 0.4546161277342834}

In [6]:
r['contribution_per_eligible']

{'control_mean': 0.06457099320391309,
 'treatment_mean': 0.017699975620162126,
 'diff': -0.046871017583750965,
 'ci': [-0.06781193329869266, -0.025930101868809268],
 'p_value': 1.1529281173886956e-05,
 'boot_ci': [-0.06780066954999087, -0.027633055899974723]}

In [7]:
r['decision']

{'outcome_code': 3,
 'decision': 'NO RELIABLE IMPROVEMENT: do not launch (and contribution profit is significantly lower)',
 'recommended_action': 'No statistically reliable conversion lift at this sample size. Do not spend on the campaign; if the idea is still attractive, re-test with a design powered for the minimum lift that would be profitable.',
 'conversion_significant': False,
 'profit_ci_lower': -0.06781193329869266,
 'profit_ci_upper': -0.025930101868809268}

## 3. Break-even and sensitivity (closed form, no simulation noise)

In [8]:
r['break_even']

{'break_even_rel_lift': 2.625252816584859,
 'break_even_conversion': 0.011340913088135118,
 'break_even_abs_lift': 0.008212603515833692,
 'control_unit_contribution': 17.394177461139893,
 'treated_unit_contribution': 4.798059153713297}

In [9]:
r['sensitivity']['sens_margin_x_discount'].pivot(index='contribution_margin', columns='discount_pct', values='break_even_rel_lift').round(2)

discount_pct,0.03,0.05,0.10,0.15,0.20
contribution_margin,,,,,
0.10,0.52,1.31,inf,inf,inf
0.15,0.28,0.57,2.63,inf,inf
0.20,0.19,0.36,1.14,3.94,inf
0.25,0.14,0.27,0.72,1.70,5.25
0.30,0.12,0.21,0.53,1.09,2.27


In [10]:
r['sensitivity']['sens_breakeven_discount']

,relative_lift,break_even_discount
0,0.05,0.006576
1,0.10,0.012554
2,0.15,0.018012
3,0.25,0.027618
4,0.50,0.046031
5,1.00,0.069046


## 4. Power

In [11]:
pd.DataFrame(r['power']['sample_size_table'])

,relative_lift,n_per_arm_for_80pct_power
0,0.100000,524858
1,0.150000,238651
2,0.250000,89728
3,0.500000,24740
4,1.000000,7279
5,2.625253,1525


In [12]:
{k: v for k, v in r['power'].items() if k != 'sample_size_table'}

{'assumed_rel_lift': 0.15000000000000002,
 'n_per_arm_actual': 32813,
 'n_per_arm_needed_for_assumed_lift': 238651,
 'power_at_actual_n': 0.1798461007732999,
 'mde_rel_lift_at_actual_n': 0.42839938993112736,
 'profit_mde_per_eligible': 0.02992013944306064}

## 5. How do conclusions change with the assumptions?

In [13]:
rows = []
for el in [0.5, 1.5, 5, 15, 30]:
    for mult in [1, 8]:
        c = replace(cfg, price_elasticity=el, n_eligible=int(cal['eligible_population'] * mult))
        x = analyze(c, cal, n_boot=0)
        rows.append({'elasticity': el, 'assumed_rel_lift': c.relative_lift, 'n_eligible': c.n_eligible,
                     'conv_p': round(x['conversion']['p_value'], 4),
                     'profit_diff_per_eligible': round(x['contribution_per_eligible']['diff'], 4),
                     'decision': x['decision']['outcome_code']})
pd.DataFrame(rows)

,elasticity,assumed_rel_lift,n_eligible,conv_p,profit_diff_per_eligible,decision
0,0.5,0.05,65627,0.9998,-0.0481,3
1,0.5,0.05,525016,0.7717,-0.0467,3
2,1.5,0.15,65627,0.4546,-0.0469,3
3,1.5,0.15,525016,0.0936,-0.0453,3
4,5.0,0.50,65627,0.0017,-0.0401,2
5,5.0,0.50,525016,0.0000,-0.0400,2
6,15.0,1.50,65627,0.0000,-0.0252,2
7,15.0,1.50,525016,0.0000,-0.0242,2
8,30.0,3.00,65627,0.0000,0.0020,2
9,30.0,3.00,525016,0.0000,-0.0002,2


## 6. Monte Carlo (200 re-runs of the default design)

In [14]:
mc = pd.read_csv(ROOT/'reports/results/experiment_monte_carlo.csv'); mc.outcome_code.value_counts(normalize=True)

outcome_code
3    0.84
2    0.16
Name: proportion, dtype: float64